# Module 1: 16S Pipeline with DADA2 and QIIME2

In this notebook we will process 16S rRNA reads from raw FASTQ files to obtain:
- An ASV table
- Taxonomic assignment
- Alpha and beta diversity metrics
- Taxonomic plots

## Pipeline overview

```
Raw FASTQ
    └─► Import into QIIME2
            └─► Visualize quality
                    └─► Remove primers (cutadapt)
                            └─► Denoising (DADA2)
                                    └─► Phylogeny
                                            └─► Alpha and beta diversity
                                                    └─► Taxonomy
```

## Requirements

Make sure the environment is active before opening this notebook:

```bash
conda activate qiime2-amplicon-2025.4
jupyter notebook
```

> **Important:** Always run **Section 0 (Path configuration)** first before running any other cell. If you restart the kernel, re-run that cell before continuing.

## 0. Path configuration

We define all paths in one place to make it easy to adapt this notebook to a different dataset.

If your computer has limited RAM, set `USE_SMALL_DATASET = True` in the next cell. Small mode uses four samples with 500 read pairs each, writes outputs to `results_small/`, runs QIIME2 commands with one thread, uses lighter non-phylogenetic core diversity metrics, and skips the memory-heavy taxonomy classifier by default.

In [ ]:
# Low-memory dataset switch
# Set to True for laptops with ~8 GB RAM or for a quick classroom demo.
# Small mode uses 4 samples x 500 read pairs and writes to results_small/.
USE_SMALL_DATASET = False

# The Silva sklearn classifier can be memory-heavy. By default, taxonomy is
# skipped in small mode and enabled in full mode. Set True to force taxonomy.
RUN_TAXONOMY = not USE_SMALL_DATASET

In [1]:
# Safety check — re-runs setup if variables are missing (e.g. after kernel restart)
try:
    METADATA
    print("✓ Variables already defined")
except NameError:
    import os
    try:
        USE_SMALL_DATASET
    except NameError:
        USE_SMALL_DATASET = False
    try:
        RUN_TAXONOMY
    except NameError:
        RUN_TAXONOMY = not USE_SMALL_DATASET

    conda_prefix = os.environ.get("CONDA_PREFIX", "")
    if conda_prefix:
        os.environ["PATH"] = os.path.join(conda_prefix, "bin") + ":" + os.environ["PATH"]
    r_home = os.path.join(conda_prefix, "lib", "R")
    if os.path.exists(r_home):
        os.environ["R_HOME"] = r_home
    _cwd = os.getcwd()
    _module_dir = None
    for _c in [_cwd, os.path.join(_cwd, "modulo1_16S_DADA2_QIIME2"),
               os.path.dirname(_cwd), os.path.join(os.path.dirname(_cwd), "modulo1_16S_DADA2_QIIME2")]:
        if os.path.exists(os.path.join(_c, "data", "metadata.tsv")):
            _module_dir = os.path.abspath(_c)
            break
    if _module_dir is None:
        raise RuntimeError("Cannot find module directory — launch Jupyter from the repo root.")

    if USE_SMALL_DATASET:
        DATA_DIR    = os.path.join(_module_dir, "data", "raw_reads_small")
        RESULTS_DIR = os.path.join(_module_dir, "results_small")
        METADATA    = os.path.join(_module_dir, "data", "metadata_small.tsv")
        MANIFEST    = os.path.join(_module_dir, "data", "manifest_small.tsv")
        QIIME_THREADS = 1
        DADA2_N_READS_LEARN = 2000
    else:
        DATA_DIR    = os.path.join(_module_dir, "data", "raw_reads")
        RESULTS_DIR = os.path.join(_module_dir, "results")
        METADATA    = os.path.join(_module_dir, "data", "metadata.tsv")
        MANIFEST    = os.path.join(_module_dir, "data", "manifest.tsv")
        QIIME_THREADS = 4
        DADA2_N_READS_LEARN = 1000000

    CLASSIFIER  = os.path.join(_module_dir, "data", "taxonomy_db", "silva-138-99-nb-classifier.qza")
    FWD_PRIMER  = "GTGCCAGCMGCCGCGGTAA";  REV_PRIMER  = "GGACTACHVGGGTWTCTAAT"
    FWD_ADAPTER = "ATTAGAWACCCBDGTAGTCC"; REV_ADAPTER = "TTACCGCGGCKGCTGGCAC"
    os.makedirs(RESULTS_DIR, exist_ok=True)
    mode = "small low-memory" if USE_SMALL_DATASET else "full"
    print(f"✓ Variables restored ({mode} mode)  |  Metadata exists: {os.path.exists(METADATA)}")

✓ Variables restored  |  Metadata exists: True


In [2]:
import os

# Prepend conda env bin to PATH so the correct R is used
conda_prefix = os.environ.get("CONDA_PREFIX", "")
if conda_prefix:
    os.environ["PATH"] = os.path.join(conda_prefix, "bin") + ":" + os.environ["PATH"]

# Fix R_HOME for Apple Silicon
r_home = os.path.join(conda_prefix, "lib", "R")
if os.path.exists(r_home):
    os.environ["R_HOME"] = r_home

# --- Find module directory using absolute paths ---
# Works regardless of where Jupyter was launched (repo root, notebook dir, etc.)
_cwd = os.getcwd()
_module_dir = None
for _candidate in [
    _cwd,
    os.path.join(_cwd, "modulo1_16S_DADA2_QIIME2"),
    os.path.dirname(_cwd),
    os.path.join(os.path.dirname(_cwd), "modulo1_16S_DADA2_QIIME2"),
]:
    if os.path.exists(os.path.join(_candidate, "data", "metadata.tsv")):
        _module_dir = os.path.abspath(_candidate)
        break

if _module_dir is None:
    raise RuntimeError(
        "Cannot find the module directory. "
        "Launch Jupyter from the repository root: clases-sistemas-microbiologicos/"
    )

# --- Absolute paths (safe regardless of working directory) ---
# USE_SMALL_DATASET is defined in the cell above. Small mode is intended for
# laptops with limited RAM and quick demonstrations.
if USE_SMALL_DATASET:
    DATA_DIR    = os.path.join(_module_dir, "data", "raw_reads_small")
    RESULTS_DIR = os.path.join(_module_dir, "results_small")
    METADATA    = os.path.join(_module_dir, "data", "metadata_small.tsv")
    MANIFEST    = os.path.join(_module_dir, "data", "manifest_small.tsv")
    QIIME_THREADS = 1
    DADA2_N_READS_LEARN = 2000
else:
    DATA_DIR    = os.path.join(_module_dir, "data", "raw_reads")
    RESULTS_DIR = os.path.join(_module_dir, "results")
    METADATA    = os.path.join(_module_dir, "data", "metadata.tsv")
    MANIFEST    = os.path.join(_module_dir, "data", "manifest.tsv")
    QIIME_THREADS = 4
    DADA2_N_READS_LEARN = 1000000

CLASSIFIER  = os.path.join(_module_dir, "data", "taxonomy_db", "silva-138-99-nb-classifier.qza")

# --- 16S Primers (V4 region, 515F/806R) ---
FWD_PRIMER  = "GTGCCAGCMGCCGCGGTAA"    # 515F (forward)   → -g in cutadapt
REV_PRIMER  = "GGACTACHVGGGTWTCTAAT"   # 806R (reverse)   → -G in cutadapt
FWD_ADAPTER = "ATTAGAWACCCBDGTAGTCC"   # RC of 806R (3' adapter on R1) → -a
REV_ADAPTER = "TTACCGCGGCKGCTGGCAC"    # RC of 515F (3' adapter on R2) → -A

os.makedirs(RESULTS_DIR, exist_ok=True)

mode = "small low-memory" if USE_SMALL_DATASET else "full"
print(f"Dataset mode     : {mode}")
print(f"Module directory : {_module_dir}")
print(f"Raw reads        : {DATA_DIR}  (exists: {os.path.exists(DATA_DIR)})")
print(f"Metadata         : {METADATA}  (exists: {os.path.exists(METADATA)})")
print(f"Results          : {RESULTS_DIR}")
print(f"QIIME threads    : {QIIME_THREADS}")
print(f"DADA2 reads learn: {DADA2_N_READS_LEARN:,}")
print(f"Run taxonomy     : {RUN_TAXONOMY}")
print(f"R                : {os.popen('which R').read().strip()}")
print(f"R version        : {os.popen('R --version 2>&1 | head -1').read().strip()}")

Module directory : /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2
Metadata         : /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/data/metadata.tsv  (exists: True)
R                : /Users/fmelis/micromamba/envs/qiime2-amplicon-2025.4/bin/R
R version        : R version 4.3.3 (2024-02-29) -- "Angel Food Cake"


## 1. Manifest and metadata

QIIME2 needs a **manifest** to locate the FASTQ files for each sample, and a **metadata** file with information about the samples.

### Manifest
Three columns: `sample-id`, `forward-absolute-filepath`, `reverse-absolute-filepath`.

### Metadata
One row per sample with variables of interest (treatment, timepoint, etc.).

The manifest is generated automatically from whichever read directory was selected in step 0:
- Full mode: `data/raw_reads/` and `data/metadata.tsv`
- Small mode: `data/raw_reads_small/` and `data/metadata_small.tsv`

In [3]:
import pandas as pd
import glob

# --- Generate manifest ---
raw_reads_abs = os.path.abspath(DATA_DIR)
r1_files = sorted(glob.glob(os.path.join(raw_reads_abs, "*_R1.fastq.gz")))
if not r1_files:
    raise FileNotFoundError(f"No *_R1.fastq.gz files found in {raw_reads_abs}")

rows = []
for r1 in r1_files:
    sample_id = os.path.basename(r1).replace("_R1.fastq.gz", "")
    r2 = r1.replace("_R1.fastq.gz", "_R2.fastq.gz")
    if not os.path.exists(r2):
        raise FileNotFoundError(f"Missing reverse read for {sample_id}: {r2}")
    rows.append({"sample-id": sample_id,
                 "forward-absolute-filepath": r1,
                 "reverse-absolute-filepath": r2})

manifest = pd.DataFrame(rows)
manifest.to_csv(MANIFEST, sep="	", index=False)
print(f"Manifest generated for {len(manifest)} samples:")
manifest

Manifest generated:


,sample-id,forward-absolute-filepath,reverse-absolute-filepath
0,sample1,/Users/fmelis/Documents/Colabs/clases_sistemas...,/Users/fmelis/Documents/Colabs/clases_sistemas...
1,sample2,/Users/fmelis/Documents/Colabs/clases_sistemas...,/Users/fmelis/Documents/Colabs/clases_sistemas...
2,sample3,/Users/fmelis/Documents/Colabs/clases_sistemas...,/Users/fmelis/Documents/Colabs/clases_sistemas...
3,sample4,/Users/fmelis/Documents/Colabs/clases_sistemas...,/Users/fmelis/Documents/Colabs/clases_sistemas...


In [4]:
metadata = pd.read_csv(METADATA, sep="\t")
print("Metadata:")
metadata

Metadata:


,#SampleID,Subject,Treatment,Timepoint
0,sample1,Mouse1,Control,T0
1,sample2,Mouse1,Treated,T1
2,sample3,Cat1,Control,T0
3,sample4,Cat1,Treated,T1


In [5]:
!qiime tools import \
    --type 'SampleData[PairedEndSequencesWithQuality]' \
    --input-path {MANIFEST} \
    --output-path {RESULTS_DIR}/sequences.qza \
    --input-format PairedEndFastqManifestPhred33V2

print("✓ Sequences imported:", RESULTS_DIR + "/sequences.qza")

Imported /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/data/manifest.tsv as PairedEndFastqManifestPhred33V2 to /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/sequences.qza
✓ Sequences imported: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/sequences.qza


In [6]:
!qiime demux summarize \
    --i-data {RESULTS_DIR}/sequences.qza \
    --o-visualization {RESULTS_DIR}/sequences_summary.qzv

print("✓ Quality summary generated")
print("  → Visualize at https://view.qiime2.org — upload:", RESULTS_DIR + "/sequences_summary.qzv")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/sequences_summary.qzv
✓ Quality summary generated
  → Visualize at https://view.qiime2.org — upload: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/sequences_summary.qzv


### What to look for in the quality plot

- X axis: position in the read (bp)
- Y axis: Phred quality score (Q)
- **Q ≥ 30** = 99.9% accuracy (good quality)
- **Q < 20** = low quality zone, consider truncating

Use this plot to set `--p-trunc-len-f` and `--p-trunc-len-r` in the denoising step.

## 4. Remove primers with Cutadapt

Primers must be removed before denoising. If they are not removed, DADA2 may interpret them as real biological variation and generate spurious ASVs.

We run cutadapt twice (`--p-times 2`) to catch any remaining primer dimers, and discard reads where no primer is found (`--discard-untrimmed`).

In [7]:
!qiime cutadapt trim-paired \
    --i-demultiplexed-sequences {RESULTS_DIR}/sequences.qza \
    --p-front-f {FWD_PRIMER} \
    --p-front-r {REV_PRIMER} \
    --p-adapter-f {FWD_ADAPTER} \
    --p-adapter-r {REV_ADAPTER} \
    --p-times 2 \
    --p-discard-untrimmed \
    --p-cores {QIIME_THREADS} \
    --o-trimmed-sequences {RESULTS_DIR}/sequences_trimmed.qza \
    --verbose 2>&1 | tail -20

print("✓ Primers removed")

226	3	0.0	1	3
228	3	0.0	1	3
231	1	0.0	1	1


=== Second read: Adapter 4 ===

Sequence: GGACTACHVGGGTWTCTAAT; Type: regular 5'; Length: 20; Trimmed: 9883 times

Minimum overlap: 3
No. of allowed errors:
1-9 bp: 0; 10-19 bp: 1; 20 bp: 2

Overview of removed sequences
length	count	expect	max.err	error counts
18	2	0.0	1	2
19	75	0.0	1	16 56 3
20	9796	0.0	2	9308 447 41
21	10	0.0	2	2 7 1
Saved SampleData[PairedEndSequencesWithQuality] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/sequences_trimmed.qza
✓ Primers removed


In [8]:
# Visualize quality after trimming
!qiime demux summarize \
    --i-data {RESULTS_DIR}/sequences_trimmed.qza \
    --o-visualization {RESULTS_DIR}/sequences_trimmed_summary.qzv

print("✓ Visualize post-trimming quality at https://view.qiime2.org")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/sequences_trimmed_summary.qzv
✓ Visualize post-trimming quality at https://view.qiime2.org


## 5. Denoising with DADA2

DADA2 does four things in a single step:
1. **Filters** low-quality reads
2. **Learns** the error model of the run
3. **Denoises** and corrects sequencing errors
4. **Removes** chimeras

The result is a set of **ASVs** (Amplicon Sequence Variants) — exact sequences, more precise than OTUs.

### ASVs vs OTUs
| | OTUs | ASVs |
|---|---|---|
| Similarity | 97% | 100% (exact sequence) |
| Resolution | Genus/species | Sub-species |
| Reproducibility | Depends on threshold | High |
| Method | Clustering | Denoising |

### Key parameters
- `--p-trunc-len-f` / `--p-trunc-len-r`: position to truncate reads. Use `0` to skip truncation (DADA2 handles quality internally).
- If quality drops sharply before the end of the read, truncate there — but make sure R1 and R2 still overlap by at least 20bp.

In [9]:
# trunc-len 0 = no truncation (DADA2 filters by quality internally)
# Adjust if quality drops before the end of the read (see step 3)
TRUNC_F = 0
TRUNC_R = 0

!qiime dada2 denoise-paired \
    --i-demultiplexed-seqs {RESULTS_DIR}/sequences_trimmed.qza \
    --p-trim-left-f 0 \
    --p-trim-left-r 0 \
    --p-trunc-len-f {TRUNC_F} \
    --p-trunc-len-r {TRUNC_R} \
    --p-n-threads {QIIME_THREADS} \
    --p-n-reads-learn {DADA2_N_READS_LEARN} \
    --o-table {RESULTS_DIR}/asv_table.qza \
    --o-representative-sequences {RESULTS_DIR}/rep_seqs.qza \
    --o-denoising-stats {RESULTS_DIR}/denoising_stats.qza

print("✓ Denoising complete")

Saved FeatureTable[Frequency] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/asv_table.qza
Saved FeatureData[Sequence] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/rep_seqs.qza
Saved SampleData[DADA2Stats] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/denoising_stats.qza
✓ Denoising complete


## 6. Denoising statistics

Check how many reads passed each filtering step. If many reads are lost at a particular step, it may indicate a problem with the parameters.

In [10]:
!qiime metadata tabulate \
    --m-input-file {RESULTS_DIR}/denoising_stats.qza \
    --o-visualization {RESULTS_DIR}/denoising_stats.qzv

print("✓ Denoising statistics generated")
print("  → Visualize at https://view.qiime2.org")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/denoising_stats.qzv
✓ Denoising statistics generated
  → Visualize at https://view.qiime2.org


### What to expect

| Column | Description | Expected value |
|---|---|---|
| `input` | Total reads | 100% |
| `filtered` | Post quality filter | > 80% |
| `denoised` | Post error correction | ~ filtered |
| `merged` | R1+R2 merged | > 70% |
| `non-chimeric` | Post chimera removal | > 70% of input |

## 7. ASV table and representative sequences

Visualize the ASV table to see how many ASVs and reads we have per sample.

In [11]:
!qiime feature-table summarize \
    --i-table {RESULTS_DIR}/asv_table.qza \
    --m-sample-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/asv_table_summary.qzv

!qiime feature-table tabulate-seqs \
    --i-data {RESULTS_DIR}/rep_seqs.qza \
    --o-visualization {RESULTS_DIR}/rep_seqs.qzv

print("✓ ASV table and representative sequences generated")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/asv_table_summary.qzv
Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/rep_seqs.qzv
✓ ASV table and representative sequences generated


## 8. Phylogeny

We build a phylogenetic tree from the representative sequences. This is required for diversity metrics that account for evolutionary distances (UniFrac).

The `align-to-tree-mafft-fasttree` command does in one step:
1. Multiple sequence alignment with MAFFT
2. Mask uninformative positions
3. Build tree with FastTree
4. Root the tree

In [12]:
!qiime phylogeny align-to-tree-mafft-fasttree \
    --i-sequences {RESULTS_DIR}/rep_seqs.qza \
    --o-alignment {RESULTS_DIR}/aligned_rep_seqs.qza \
    --o-masked-alignment {RESULTS_DIR}/masked_aligned_rep_seqs.qza \
    --o-tree {RESULTS_DIR}/unrooted_tree.qza \
    --o-rooted-tree {RESULTS_DIR}/rooted_tree.qza

print("✓ Phylogenetic tree built")

Saved FeatureData[AlignedSequence] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/aligned_rep_seqs.qza
Saved FeatureData[AlignedSequence] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/masked_aligned_rep_seqs.qza
Saved Phylogeny[Unrooted] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/unrooted_tree.qza
Saved Phylogeny[Rooted] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/rooted_tree.qza
✓ Phylogenetic tree built


## 9. Alpha and beta diversity

### Alpha diversity
Measures diversity **within** a single sample.
- **Shannon:** accounts for richness and relative abundance
- **Observed features:** number of unique ASVs
- **Faith's PD:** phylogenetic diversity

### Beta diversity
Measures diversity **between** samples.
- **Bray-Curtis:** abundance-based, no phylogeny
- **Unweighted UniFrac:** presence/absence + phylogeny
- **Weighted UniFrac:** abundance + phylogeny

### Rarefaction
Before computing diversity, we normalize by sequencing depth (rarefaction). The `--p-sampling-depth` parameter sets the minimum depth — samples with fewer reads will be excluded.

Check the ASV table summary to choose a value that retains most samples.

In [13]:
import zipfile, pandas as pd

# Auto-detect sampling depth from denoising stats
with zipfile.ZipFile(f"{RESULTS_DIR}/denoising_stats.qza") as z:
    tsv = [f for f in z.namelist() if f.endswith(".tsv") and "stats" in f][0]
    with z.open(tsv) as f:
        # Skip first comment line, use second line as header
        lines = [l.decode() for l in f.readlines()]

header = lines[1].strip().split("\t")
print("Columns:", header)

counts = {}
for line in lines[2:]:
    parts = line.strip().split("\t")
    if len(parts) >= 2:
        sample = parts[0]
        # Find 'non-chimeric' column by name
        try:
            idx = header.index("non-chimeric")
            counts[sample] = int(parts[idx])
        except (ValueError, IndexError):
            # Fallback: last integer column
            for val in reversed(parts):
                try:
                    counts[sample] = int(val); break
                except ValueError:
                    continue

print("\nReads per sample after DADA2:")
for sample, count in counts.items():
    print(f"  {sample}: {count:,}")

min_count = min(counts.values())
SAMPLING_DEPTH = max((int(min_count * 0.8) // 100) * 100, 100)
print(f"\nMin reads: {min_count:,}")
print(f"Sampling depth set to: {SAMPLING_DEPTH:,}  (80% of min — adjust if needed)")

Columns: ['#q2:types', 'numeric', 'numeric', 'numeric', 'numeric', 'numeric', 'numeric', 'numeric', 'numeric']

Reads per sample after DADA2:
  sample1: 3,560
  sample2: 3,820
  sample3: 3,970
  sample4: 4,437

Min reads: 3,560
Sampling depth set to: 2,800  (80% of min — adjust if needed)


In [14]:
# SAMPLING_DEPTH is set automatically by the cell above
# Small mode uses non-phylogenetic core metrics to reduce runtime and memory.
import subprocess

if USE_SMALL_DATASET:
    cmd = [
        "qiime", "diversity", "core-metrics",
        "--i-table", f"{RESULTS_DIR}/asv_table.qza",
        "--p-sampling-depth", str(SAMPLING_DEPTH),
        "--m-metadata-file", METADATA,
        "--output-dir", f"{RESULTS_DIR}/diversity",
    ]
else:
    cmd = [
        "qiime", "diversity", "core-metrics-phylogenetic",
        "--i-phylogeny", f"{RESULTS_DIR}/rooted_tree.qza",
        "--i-table", f"{RESULTS_DIR}/asv_table.qza",
        "--p-sampling-depth", str(SAMPLING_DEPTH),
        "--m-metadata-file", METADATA,
        "--output-dir", f"{RESULTS_DIR}/diversity",
    ]

subprocess.run(cmd, check=True)
print("✓ Diversity metrics computed in:", RESULTS_DIR + "/diversity/")

Saved FeatureTable[Frequency] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/diversity/rarefied_table.qza
Saved SampleData[AlphaDiversity] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/diversity/faith_pd_vector.qza
Saved SampleData[AlphaDiversity] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/diversity/observed_features_vector.qza
Saved SampleData[AlphaDiversity] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/diversity/shannon_vector.qza
Saved SampleData[AlphaDiversity] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/diversity/evenness_vector.qza
Saved DistanceMatrix 

In [15]:
# Rarefaction curves — check at what depth diversity stabilizes
# In small mode, skip phylogenetic rarefaction and use fewer curve steps.
import subprocess

cmd = [
    "qiime", "diversity", "alpha-rarefaction",
    "--i-table", f"{RESULTS_DIR}/asv_table.qza",
    "--p-max-depth", str(SAMPLING_DEPTH),
    "--p-steps", "4" if USE_SMALL_DATASET else "10",
    "--m-metadata-file", METADATA,
    "--o-visualization", f"{RESULTS_DIR}/alpha_rarefaction.qzv",
]
if not USE_SMALL_DATASET:
    cmd.extend(["--i-phylogeny", f"{RESULTS_DIR}/rooted_tree.qza"])

subprocess.run(cmd, check=True)
print("✓ Rarefaction curves generated")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/alpha_rarefaction.qzv
✓ Rarefaction curves generated


In [16]:
# Statistical test: differences in alpha diversity between groups
!qiime diversity alpha-group-significance \
    --i-alpha-diversity {RESULTS_DIR}/diversity/shannon_vector.qza \
    --m-metadata-file {METADATA} \
    --o-visualization {RESULTS_DIR}/diversity/shannon_significance.qzv

print("✓ Alpha diversity significance test (Shannon) generated")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/diversity/shannon_significance.qzv
✓ Alpha diversity significance test (Shannon) generated


In [17]:
# Statistical test: differences in community composition between groups (PERMANOVA)
!qiime diversity beta-group-significance \
    --i-distance-matrix {RESULTS_DIR}/diversity/bray_curtis_distance_matrix.qza \
    --m-metadata-file {METADATA} \
    --m-metadata-column Treatment \
    --o-visualization {RESULTS_DIR}/diversity/bray_curtis_significance.qzv

print("✓ PERMANOVA test (Bray-Curtis) generated")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/diversity/bray_curtis_significance.qzv
✓ PERMANOVA test (Bray-Curtis) generated


### PCoA — Sample ordination

PCoA (Principal Coordinates Analysis) displays sample similarity in a reduced space. Samples that cluster together have similar microbial composition.

The `*_emperor.qzv` files in the diversity folder contain the interactive PCoA plots.

In [18]:
import os
pcoa_files = [f for f in os.listdir(RESULTS_DIR + "/diversity") if "emperor" in f]
print("Available PCoA files:")
for f in pcoa_files:
    print(" →", f)
print("\nVisualize at https://view.qiime2.org")

Available PCoA files:
 → unweighted_unifrac_emperor.qzv
 → jaccard_emperor.qzv
 → bray_curtis_emperor.qzv
 → weighted_unifrac_emperor.qzv

Visualize at https://view.qiime2.org


## 10. Taxonomic assignment

We assign taxonomy to each ASV using a Naive Bayes classifier trained on the Silva 138 database.

The classifier file is ~1.7GB and must be downloaded once before running this step. The classifier can be memory-heavy, so `RUN_TAXONOMY` is set to `False` automatically when `USE_SMALL_DATASET = True`. If you want to run taxonomy in small mode, set `RUN_TAXONOMY = True` in Section 0 and rerun from there.

In [19]:
import subprocess

if RUN_TAXONOMY:
    if not os.path.exists(CLASSIFIER):
        print("Downloading Silva classifier (~1.7GB) — this may take a few minutes...")
        subprocess.run([
            "wget", "-q", "--show-progress", "-O", CLASSIFIER,
            "https://data.qiime2.org/classifiers/sklearn-1.4.2/silva/silva-138-99-nb-classifier.qza",
        ], check=True)
        print("✓ Silva classifier downloaded")
    else:
        print("✓ Silva classifier already exists, skipping download")
else:
    print("Skipping taxonomy download because RUN_TAXONOMY = False")

✓ Silva classifier already exists, skipping download


In [20]:
import subprocess

if RUN_TAXONOMY:
    subprocess.run([
        "qiime", "feature-classifier", "classify-sklearn",
        "--i-classifier", CLASSIFIER,
        "--i-reads", f"{RESULTS_DIR}/rep_seqs.qza",
        "--p-n-jobs", str(QIIME_THREADS),
        "--o-classification", f"{RESULTS_DIR}/taxonomy.qza",
    ], check=True)
    print("✓ Taxonomy assigned")
else:
    print("Skipping taxonomy classification because RUN_TAXONOMY = False")

Saved FeatureData[Taxonomy] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/taxonomy.qza
✓ Taxonomy assigned


In [21]:
import subprocess

if RUN_TAXONOMY:
    subprocess.run([
        "qiime", "metadata", "tabulate",
        "--m-input-file", f"{RESULTS_DIR}/taxonomy.qza",
        "--o-visualization", f"{RESULTS_DIR}/taxonomy.qzv",
    ], check=True)
    print("✓ Taxonomy table generated")
else:
    print("Skipping taxonomy table because RUN_TAXONOMY = False")

Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/taxonomy.qzv
✓ Taxonomy table generated


## 11. Filter contaminants and plot relative abundance (barplots)

Before visualizing taxonomic composition, we remove ASVs assigned to **mitochondria** and **chloroplasts** — these are eukaryotic 16S sequences that do not represent bacterial microbiota.

Barplots show the relative abundance of each sample. You can change the taxonomic level (Phylum, Class, Order, Family, Genus) in the interactive viewer.

In [22]:
import subprocess

if RUN_TAXONOMY:
    subprocess.run([
        "qiime", "taxa", "filter-table",
        "--i-table", f"{RESULTS_DIR}/asv_table.qza",
        "--i-taxonomy", f"{RESULTS_DIR}/taxonomy.qza",
        "--p-exclude", "mitochondria,chloroplast",
        "--o-filtered-table", f"{RESULTS_DIR}/asv_table_filtered.qza",
    ], check=True)
    print("✓ Mitochondria and chloroplasts removed")

    subprocess.run([
        "qiime", "taxa", "barplot",
        "--i-table", f"{RESULTS_DIR}/asv_table_filtered.qza",
        "--i-taxonomy", f"{RESULTS_DIR}/taxonomy.qza",
        "--m-metadata-file", METADATA,
        "--o-visualization", f"{RESULTS_DIR}/taxa_barplot.qzv",
    ], check=True)
    print("✓ Taxonomic barplot generated")
    print("  → Visualize at https://view.qiime2.org")
else:
    print("Skipping contaminant filtering and taxonomic barplot because RUN_TAXONOMY = False")

Saved FeatureTable[Frequency] to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/asv_table_filtered.qza
✓ Mitochondria and chloroplasts removed
Saved Visualization to: /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results/taxa_barplot.qzv
✓ Taxonomic barplot generated
  → Visualize at https://view.qiime2.org


## 12. Summary of generated files

At the end of the pipeline you should have the following files in `results/`:

In [23]:
import os

print(f"Contents of {RESULTS_DIR}:\n")
for root, dirs, files in os.walk(RESULTS_DIR):
    level = root.replace(RESULTS_DIR, '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    if level > 0:
        print(f"{indent}{folder}/")
    for f in sorted(files):
        print(f"{indent}  {f}")

Contents of /Users/fmelis/Documents/Colabs/clases_sistemas_microbiologicos/clases-sistemas-microbiologicos/modulo1_16S_DADA2_QIIME2/results:

  .gitkeep
  aligned_rep_seqs.qza
  alpha_rarefaction.qzv
  asv_table.qza
  asv_table_filtered.qza
  asv_table_summary.qzv
  denoising_stats.qza
  denoising_stats.qzv
  masked_aligned_rep_seqs.qza
  rep_seqs.qza
  rep_seqs.qzv
  rooted_tree.qza
  sequences.qza
  sequences_summary.qzv
  sequences_trimmed.qza
  sequences_trimmed_summary.qzv
  taxa_barplot.qzv
  taxonomy.qza
  taxonomy.qzv
  unrooted_tree.qza
  diversity/
    bray_curtis_distance_matrix.qza
    bray_curtis_emperor.qzv
    bray_curtis_pcoa_results.qza
    bray_curtis_significance.qzv
    evenness_vector.qza
    faith_pd_vector.qza
    jaccard_distance_matrix.qza
    jaccard_emperor.qzv
    jaccard_pcoa_results.qza
    observed_features_vector.qza
    rarefied_table.qza
    shannon_significance.qzv
    shannon_vector.qza
    unweighted_unifrac_distance_matrix.qza
    unweighted_unifra

## Quick reference: `.qza` vs `.qzv` files

| Extension | Type | Use |
|---|---|---|
| `.qza` | Artifact | Processed data, input for next steps |
| `.qzv` | Visualization | View only at view.qiime2.org |

Both are ZIP files — you can rename them to `.zip` and open them to inspect their contents.